# S2 — Baseline classifier (real data only)

**Goal:** measure how well a classifier trained on **real images only** recognises each class, especially the rare class **df**. This is the reference that synthetic images must beat in S5. The best model is also kept as the **feature extractor** for the generative-model evaluation.

**Metrics** (on the real test set): **precision, recall and F1** per class and macro-averaged, plus the **confusion matrix**.

**Runs:** 3 strategies × 3 seeds (from `configs/classifier.yaml`):

| strategy | idea |
|---|---|
| `none` | plain cross-entropy: the reference baseline |
| `weighted` | cross-entropy weighted by inverse class frequency |
| `oversample` | the sampler draws rare classes more often |

The best epoch of each run is chosen on the **validation** macro-F1; the test set is used only once at the end.

> Use a GPU (Colab: *Runtime → Change runtime type → GPU*). On a laptop CPU, set `QUICK = True` for a short test.

In [ ]:
# --- Setup: works locally (.venv) and on Google Colab ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()   # browser or VS Code Colab kernel
if IN_COLAB:
    REPO = "https://github.com/zouhairi1/mini_project_generative_visuelle.git"
    target = Path("/content/mini_project_generative_visuelle")
    if not target.exists():
        subprocess.run(["git", "clone", REPO, str(target)], check=True)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")                      # run from the repository root

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Project root:", Path.cwd())

In [ ]:
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data import load_split, SHORT_NAMES, SPLITS
from src.train_clf import train_classifier, evaluate, summarize
from src.viz import plot_confusion_matrix
from src.utils import load_config, get_device, ensure_dirs, RESULTS_DIR, CKPT_DIR

QUICK = os.environ.get("QUICK", "0") == "1"   # set QUICK = True for a fast smoke test (1 seed, 2 epochs)

cfg = load_config("classifier")
if QUICK:
    cfg["train"].update(epochs=2, seeds=[0], strategies=["none"])
RARE = cfg["rare_class"]
ensure_dirs()
device = get_device()
print("device:", device, "| train config:", cfg["train"])

(train_x, train_y), (val_x, val_y), (test_x, test_y) = [load_split(s, cfg["data"]["size"]) for s in SPLITS]
print("train", train_x.shape, "| val", val_x.shape, "| test", test_x.shape)

## 1. Training

In [ ]:
runs = []
for strategy in cfg["train"]["strategies"]:
    for seed in cfg["train"]["seeds"]:
        model, history = train_classifier(train_x, train_y, val_x, val_y, cfg,
                                          seed=seed, strategy=strategy, device=device)
        runs.append({
            "strategy": strategy,
            "seed": seed,
            "history": history,
            "val": evaluate(model, val_x, val_y, device),
            "test": evaluate(model, test_x, test_y, device),
        })
        torch.save(model.state_dict(), CKPT_DIR / f"clf_{strategy}_seed{seed}.pt")

with open(RESULTS_DIR / "s2_baseline_runs.json", "w") as f:
    json.dump(runs, f, indent=1)

# Reference model (strategy 'none', best validation macro-F1): feature extractor for S5
ref = max((r for r in runs if r["strategy"] == "none"), key=lambda r: r["val"]["macro_f1"])
shutil.copy(CKPT_DIR / f"clf_none_seed{ref['seed']}.pt", CKPT_DIR / "baseline_classifier.pt")
print("Reference model: strategy 'none', seed", ref["seed"], "-> checkpoints/baseline_classifier.pt")

## 2. Results on the test set (mean ± std over seeds)

In [ ]:
rows = []
for strategy in cfg["train"]["strategies"]:
    rs = [r["test"] for r in runs if r["strategy"] == strategy]
    rows.append({
        "strategy": strategy,
        "macro precision": summarize([m["macro_precision"] for m in rs]),
        "macro recall": summarize([m["macro_recall"] for m in rs]),
        "macro F1": summarize([m["macro_f1"] for m in rs]),
        "df precision": summarize([m["precision"][RARE] for m in rs]),
        "df recall": summarize([m["recall"][RARE] for m in rs]),
        "df F1": summarize([m["f1"][RARE] for m in rs]),
    })
summary = pd.DataFrame(rows).set_index("strategy")
summary.to_csv(RESULTS_DIR / "s2_summary.csv")
summary

In [ ]:
# Per-class precision / recall / F1 (mean over seeds), one block per strategy
blocks = {}
for strategy in cfg["train"]["strategies"]:
    rs = [r["test"] for r in runs if r["strategy"] == strategy]
    blocks[strategy] = pd.DataFrame({
        "precision": np.mean([m["precision"] for m in rs], axis=0),
        "recall": np.mean([m["recall"] for m in rs], axis=0),
        "F1": np.mean([m["f1"] for m in rs], axis=0),
    }, index=SHORT_NAMES)
per_class = pd.concat(blocks, axis=1).round(3)
per_class[("test", "support")] = runs[0]["test"]["support"]
per_class.to_csv(RESULTS_DIR / "s2_per_class.csv")
per_class

## 3. Confusion matrices

One matrix per strategy (the seed with the best validation macro-F1). Rows = true class, columns = predicted class. The colour is the **row-normalised** value (= recall of each class); the numbers are image counts.

In [ ]:
strategies = cfg["train"]["strategies"]
fig, axes = plt.subplots(1, len(strategies), figsize=(5.5 * len(strategies), 5), squeeze=False)
for ax, strategy in zip(axes[0], strategies):
    best = max((r for r in runs if r["strategy"] == strategy), key=lambda r: r["val"]["macro_f1"])
    m = best["test"]
    plot_confusion_matrix(m["confusion_matrix"], ax=ax,
                          title=f"{strategy} (seed {best['seed']})\nmacro-F1 {m['macro_f1']:.2f} | df recall {m['recall'][RARE]:.2f}")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "s2_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for strategy in strategies:
    hs = [r["history"] for r in runs if r["strategy"] == strategy]
    epochs = [h["epoch"] for h in hs[0]]
    for ax, key, label in zip(axes, ["train_loss", "val_macro_f1", "val_rare_recall"],
                              ["train loss", "validation macro-F1", "validation df recall"]):
        ax.plot(epochs, np.mean([[h[key] for h in run] for run in hs], axis=0), label=strategy)
        ax.set_title(label)
        ax.set_xlabel("epoch")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "s2_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Interpretation (to complete with your numbers)

- **Majority class:** nv should have high precision and recall: the model sees it 67 % of the time.
- **Rare class df:** with plain cross-entropy (`none`), df recall is expected to be **low** — read it in the table above. With only 23 test images, each image is about 4 points of recall, hence the standard deviation over seeds.
- **Confusions:** check in the confusion matrix where the df images go (often to nv or bkl), and which classes get confused with each other (for example mel ↔ nv).
- **weighted / oversample:** these classic fixes usually **raise df recall but lower df precision** (more false df). Compare the df F1 of the three strategies.
- **Reference for S5:** the numbers of `none` are the baseline **R0**, and `oversample` is the control **R1**. Synthetic images (R2, R3) will be judged against them.

## Conclusion of S2
- Reference model saved: `checkpoints/baseline_classifier.pt` (not committed; upload it to Drive).
- Tables and figures saved in `results/` (`s2_summary.csv`, `s2_per_class.csv`, `s2_confusion_matrices.png`, `s2_training_curves.png`).
- Next step: **S3**, the generative models (CVAE and cGAN).

## 6. Bring the results back to your laptop (Colab only)

On Colab (browser or VS Code), the files above are written on Google's machine. This cell zips `results/` and the reference checkpoint and downloads `s2_results.zip`. Unzip it into your local `mini_project/` folder, then commit and push the results.

In [ ]:
import zipfile

zip_path = Path("s2_results.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for p in sorted(RESULTS_DIR.glob("s2_*")):
        z.write(p, p.relative_to(Path.cwd()))
    ref_ckpt = CKPT_DIR / "baseline_classifier.pt"
    if ref_ckpt.exists():
        z.write(ref_ckpt, ref_ckpt.relative_to(Path.cwd()))
print("Created", zip_path.resolve(), "with", len(zipfile.ZipFile(zip_path).namelist()), "files")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as e:
        print("Automatic download not available here:", e)
        print("Download the file manually from the Colab file browser:", zip_path.resolve())